# Project 2 — LLM Fine-tuning: QLoRA Training

**Duration:** 5 min

## Setup QLoRA

In [ ]:
# pip install transformers peft bitsandbytes datasets trl
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset
import torch

MODEL = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb_config, device_map='auto')

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=['q_proj','v_proj']
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

> **Try it in Google Colab:** [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shastrula/ailearningclub-courses/blob/main/ai-projects-advanced/ap-project2-build.ipynb)


## Prepare Data & Train

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

ds = load_dataset('spider')

def format_example(ex):
    return f"""### Question: {ex['question']}
### Schema: {ex['db_id']}
### SQL: {ex['query']}"""

train_ds = ds['train'].map(lambda x: {'text': format_example(x)})

training_args = TrainingArguments(
    output_dir='./sql-lora',
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=50,
    save_strategy='epoch',
)

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=train_ds,
    dataset_text_field='text',
    max_seq_length=256,
    args=training_args,
)
trainer.train()
model.save_pretrained('./sql-lora-adapter')

> **💡 Tip:** ✅ Project 2 complete! QLoRA lets you fine-tune a 1B model on a free Colab T4 in ~30 minutes.

<div class="quiz" data-correct="1">
  <p class="font-semibold mb-3">❓ What does QLoRA's 4-bit quantization achieve?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4372645568" value="0">
      <span>Faster inference only</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4372645568" value="1">
      <span>Reduces model memory by ~4x enabling fine-tuning on consumer GPUs</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4372645568" value="2">
      <span>Improves model accuracy</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4372645568" value="3">
      <span>Removes the need for a GPU</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>